# Pipeline WikiRank

Ejecuta el proyecto completo de principio a fin: ingesta en HDFS → controles de calidad y cuarentena →
zona curada en Parquet → tabla auxiliar de idiomas → preguntas P1–P3 → modelo base de ML.
Los experimentos de rendimiento están aparte, en `experiments.ipynb`.

## Antes de ejecutar
1. **Entorno levantado** (ver README): `docker compose up -d` y esperar a que HDFS salga del modo seguro:
   `docker exec namenode hdfs dfsadmin -safemode get` → `Safe mode is OFF`.
   YARN no es necesario para este cuaderno (solo para MapReduce).
2. **Dataset** en `data/wikirank_20240801.csv`. La primera ejecución lo sube a `/raw/wikirank`;
   si ya está en HDFS con el tamaño correcto, no se vuelve a subir.
3. **Recursos:** Docker con al menos 8 GB de RAM y unos 35 GB de disco.

## Configuración (`config.yaml`)
Con nuestro `docker-compose.yml` no hay que cambiar nada. En otro entorno, revisar:

| Clave | Valor | Cuándo cambiarla |
|---|---|---|
| `hdfs_base` | `hdfs://namenode:9000` | Si el NameNode tiene otro nombre o puerto |
| `spark.master` | `spark://spark-master:7077` | `local[*]` para ejecutar sin clúster |
| `spark.driver_host` | `jupyter` | Nombre del contenedor donde corre el cuaderno |
| `spark.executor_memory` | `3g` | Igual o menor que la memoria del worker; `2g` si Docker tiene poca RAM |
| `spark.dfs_replication` | `2` | Igual al número de DataNodes |
| `input.local_csv` | `/home/jovyan/work/data/wikirank_20240801.csv` | Si el CSV está en otra carpeta |
| `input.expected_bytes` | `8195179141` | Si se usa otra versión del dataset |

El resto (rutas de las zonas, límites de calidad, parámetros del modelo) no depende del entorno.

## Ejecución
*Kernel → Restart Kernel and Run All Cells* (o `jupyter nbconvert --execute`, ver README).
Duración aproximada con un worker de 2 núcleos y 3 GB: **13 min**.
La última celda cierra la sesión de Spark (`spark.stop()`); hacerlo antes de abrir `experiments.ipynb`.

In [1]:
import os, time, yaml
from pyspark.sql import SparkSession, Window, functions as F

with open("/home/jovyan/work/config.yaml") as f:
    cfg = yaml.safe_load(f)

HDFS = cfg["hdfs_base"]
RAW = HDFS + cfg["paths"]["raw"]
CURATED = HDFS + cfg["paths"]["curated"]
LANG_INFO = HDFS + cfg["paths"]["language_info"]
RESULTS = HDFS + cfg["paths"]["results"]

os.environ["HADOOP_USER_NAME"] = "root"

spark = (SparkSession.builder
         .appName("wikirank-pipeline")
         .master(cfg["spark"]["master"])
         .config("spark.driver.host", cfg["spark"]["driver_host"])
         .config("spark.executor.memory", cfg["spark"]["executor_memory"])
         .config("spark.hadoop.dfs.replication", cfg["spark"]["dfs_replication"])
         .getOrCreate())
spark


In [2]:
local_csv = cfg["input"]["local_csv"]
expected = cfg["input"]["expected_bytes"]
Path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path
fs = Path(RAW).getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
dest = Path(RAW + "/" + os.path.basename(local_csv))
tmp = Path(RAW + "/_subiendo_" + os.path.basename(local_csv))

in_hdfs = fs.exists(dest)
if in_hdfs and fs.getFileStatus(dest).getLen() == expected:
    print("El CSV ya está en HDFS y está completo:", dest)

elif os.path.exists(local_csv):
    if os.path.getsize(local_csv) != expected:
        raise ValueError(f"CSV local incompleto: {os.path.getsize(local_csv)} bytes, se esperaban {expected}")
    if in_hdfs:
        print("El CSV de HDFS está incompleto: se vuelve a subir")
    t = time.time()
    fs.mkdirs(Path(RAW))
    fs.copyFromLocalFile(False, True, Path("file://" + local_csv), tmp)
    fs.delete(dest, False)
    fs.rename(tmp, dest)
    print(f"Copiado a HDFS en {time.time() - t:.0f} s:", dest)

elif in_hdfs:
    raise ValueError("El CSV de HDFS está incompleto y no hay copia en data/ para volver a subirlo")
else:
    raise FileNotFoundError("El CSV no está en HDFS ni en data/ (ver README, apartado Dataset)")


El CSV ya está en HDFS y está completo: hdfs://namenode:9000/raw/wikirank/wikirank_20240801.csv


In [3]:
schema = ("Wikipedia_article_URL STRING, Language STRING, Page_ID LONG, "
          "Title STRING, WikiRank_score DOUBLE, WikiRank_URL STRING")

raw = (spark.read.schema(schema)
         .option("header", True)
         .option("escape", '"')
         .csv(dest.toString()))
raw.printSchema()
raw.show(3, truncate=40)


root
 |-- Wikipedia_article_URL: string (nullable = true)
 |-- Language: string (nullable = true)
 |-- Page_ID: long (nullable = true)
 |-- Title: string (nullable = true)
 |-- WikiRank_score: double (nullable = true)
 |-- WikiRank_URL: string (nullable = true)

+----------------------------------------+--------+-------+-------+--------------+----------------------------------------+
|                   Wikipedia_article_URL|Language|Page_ID|  Title|WikiRank_score|                            WikiRank_URL|
+----------------------------------------+--------+-------+-------+--------------+----------------------------------------+
|https://ar.wikipedia.org/wiki/%D9%85%...|      ar|      7|    ماء|       85.4175|https://wikirank.net/ar/%D9%85%D8%A7%...|
|https://ar.wikipedia.org/wiki/%D8%B1%...|      ar|     38|رياضيات|        79.166|https://wikirank.net/ar/%D8%B1%D9%8A%...|
|https://ar.wikipedia.org/wiki/%D8%A5%...|      ar|     62|إستونيا|       90.6549|https://wikirank.net/ar/%D8%A5%D8%B

In [4]:
q = cfg["quality"]
t = time.time()

rule = (F.col("Language").isNotNull() & F.col("Page_ID").isNotNull()
        & F.col("WikiRank_score").between(q["score_min"], q["score_max"]))

ok = F.coalesce(rule, F.lit(False))

checks = raw.agg(
    F.count("*").alias("filas_total"),
    *[F.sum(F.col(c).isNull().cast("int")).alias("nulos_" + c) for c in raw.columns],
    F.sum((~F.col("WikiRank_score").between(q["score_min"], q["score_max"])).cast("int"))
     .alias("score_fuera_de_rango"),
    F.sum((~ok).cast("int")).alias("filas_invalidas"),
).first().asDict()

checks["claves_duplicadas"] = (raw.groupBy(F.lower(F.trim("Language")).alias("Language"), "Page_ID")
                                  .count().filter("count > 1").count())

print(f"Calidad calculada en {time.time() - t:.0f} s")
for k, v in checks.items():
    print(f"{k:30s} {v}")

Calidad calculada en 122 s
filas_total                    47019883
nulos_Wikipedia_article_URL    0
nulos_Language                 0
nulos_Page_ID                  0
nulos_Title                    0
nulos_WikiRank_score           0
nulos_WikiRank_URL             0
score_fuera_de_rango           0
filas_invalidas                0
claves_duplicadas              0


In [5]:
QUARANTINE = HDFS + cfg["paths"]["quarantine"]
good, bad = raw.filter(ok), raw.filter(~ok)
bad.write.mode("overwrite").parquet(QUARANTINE)

n_bad = checks["filas_invalidas"]
print("Filas en cuarentena:", n_bad)
assert n_bad / checks["filas_total"] < q["max_bad_ratio"], f"demasiadas filas malas: {n_bad}"

Filas en cuarentena: 0


In [6]:
url = F.col("Wikipedia_article_URL")
curated = (good
    .withColumn("Language", F.lower(F.trim("Language")))
    .withColumn("Title", F.coalesce(F.col("Title"), F.lit("")))
    .withColumn("url_pct_count",
                (F.length(url) - F.length(F.regexp_replace(url, "%", ""))).cast("int"))
    .drop("Wikipedia_article_URL", "WikiRank_URL"))

if checks["claves_duplicadas"] > 0:
    curated = curated.dropDuplicates(["Language", "Page_ID"])

In [7]:
part = cfg["partition_column"]
t = time.time()
(curated.repartition(part)
    .write.mode("overwrite")
    .partitionBy(part)
    .parquet(CURATED))
print(f"Escrito en {time.time() - t:.0f} s")

articles = spark.read.parquet(CURATED)
checks["filas_curadas"] = articles.count()
checks["filas_eliminadas"] = checks["filas_total"] - checks["filas_curadas"]
print("Filas curadas:", checks["filas_curadas"], "| eliminadas:", checks["filas_eliminadas"])


Escrito en 118 s
Filas curadas: 47019883 | eliminadas: 0


In [8]:
quality = spark.createDataFrame([checks])
quality.coalesce(1).write.mode("overwrite").json(RESULTS + "/quality_report")
quality.show(vertical=True)


-RECORD 0-------------------------------
 claves_duplicadas           | 0        
 filas_curadas               | 47019883 
 filas_eliminadas            | 0        
 filas_invalidas             | 0        
 filas_total                 | 47019883 
 nulos_Language              | 0        
 nulos_Page_ID               | 0        
 nulos_Title                 | 0        
 nulos_WikiRank_URL          | 0        
 nulos_WikiRank_score        | 0        
 nulos_Wikipedia_article_URL | 0        
 score_fuera_de_rango        | 0        



In [9]:
es = articles.filter("Language = 'es'")
es.explain()
es.count()


== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [Page_ID#124L,Title#125,WikiRank_score#126,url_pct_count#127,Language#128] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [isnotnull(Language#128), (Language#128 = es)], PushedFilters: [], ReadSchema: struct<Page_ID:bigint,Title:string,WikiRank_score:double,url_pct_count:int>




1905093

**Tabla Auxiliar de Idiomas**

In [10]:
%pip install langcodes==3.5.1 language_data==1.4.0

Note: you may need to restart the kernel to use updated packages.


In [11]:
import langcodes

CODE_ALIAS = {"simple": "en"}
FIX_NAME = {"sh": "Serbo-Croatian", "simple": "Simple English"}
FIX_SCRIPT = {"sh": "Latn", "zh-min-nan": "Latn"}

FAMILY = {"ar": "Afro-Asiatic", "he": "Afro-Asiatic", "az": "Turkic", "kk": "Turkic", "tr": "Turkic",
          "uz": "Turkic", "et": "Uralic", "fi": "Uralic", "hu": "Uralic", "eu": "Isolate",
          "ka": "Kartvelian", "ja": "Japonic", "ko": "Koreanic", "id": "Austronesian",
          "ms": "Austronesian", "ta": "Dravidian", "th": "Kra-Dai", "vi": "Austroasiatic",
          "zh": "Sino-Tibetan", "zh-min-nan": "Sino-Tibetan", "eo": "Constructed", "vo": "Constructed"}

for c in ["be", "bg", "ca", "cs", "da", "de", "el", "en", "es", "fa", "fr", "gl", "hi", "hr", "hy", "it",
          "la", "lt", "nl", "nn", "no", "pl", "pt", "ro", "ru", "sh", "simple", "sk", "sl", "sr", "sv", "uk", "ur"]:
    FAMILY[c] = "Indo-European"

FSI = {"da": 1, "es": 1, "fr": 1, "it": 1, "nl": 1, "nn": 1, "no": 1, "pt": 1, "ro": 1, "sv": 1,
       "de": 2, "id": 2, "ms": 2,
       "az": 3, "be": 3, "bg": 3, "cs": 3, "el": 3, "et": 3, "fa": 3, "fi": 3, "he": 3, "hi": 3, "hr": 3,
       "hu": 3, "hy": 3, "ka": 3, "kk": 3, "lt": 3, "pl": 3, "ru": 3, "sh": 3, "sk": 3, "sl": 3, "sr": 3,
       "ta": 3, "th": 3, "tr": 3, "uk": 3, "ur": 3, "uz": 3, "vi": 3,
       "ar": 4, "ja": 4, "ko": 4, "zh": 4}

def lang_row(code):
    lang = langcodes.Language.get(CODE_ALIAS.get(code, code))
    return (code,
            FIX_NAME.get(code, lang.display_name("en")),
            round(lang.speaking_population() / 1e6, 3),
            FIX_SCRIPT.get(code, lang.maximize().script),
            FAMILY.get(code),
            FSI.get(code))

codes = sorted(r["Language"] for r in articles.select("Language").distinct().collect())

lang_info = spark.createDataFrame(
    [lang_row(c) for c in codes],
    "Language STRING, language_name STRING, speakers_millions DOUBLE, script STRING, family STRING, fsi_category INT")
lang_info.show(60, truncate=False)


+----------+-----------------+-----------------+------+-------------+------------+
|Language  |language_name    |speakers_millions|script|family       |fsi_category|
+----------+-----------------+-----------------+------+-------------+------------+
|ar        |Arabic           |378.793          |Arab  |Afro-Asiatic |4           |
|az        |Azerbaijani      |33.804           |Latn  |Turkic       |3           |
|be        |Belarusian       |10.022           |Cyrl  |Indo-European|3           |
|bg        |Bulgarian        |7.684            |Cyrl  |Indo-European|3           |
|ca        |Catalan          |8.219            |Latn  |Indo-European|NULL        |
|cs        |Czech            |13.236           |Latn  |Indo-European|3           |
|da        |Danish           |7.248            |Latn  |Indo-European|1           |
|de        |German           |141.873          |Latn  |Indo-European|2           |
|el        |Greek            |12.249           |Grek  |Indo-European|3           |
|en 

In [12]:
dups = lang_info.groupBy("Language").count().filter("count > 1").count()
assert dups == 0, f"{dups} languages are duplicated in language_info"
print(f"language_info: {lang_info.count()} languages, no duplicates")

lang_info.coalesce(1).write.mode("overwrite").parquet(LANG_INFO)
lang_info = spark.read.parquet(LANG_INFO)

language_info: 55 languages, no duplicates


**Pregunta 1:** Artículos y media por idioma

In [13]:
t = time.time()
q1 = (articles.groupBy("Language")
        .agg(F.count("*").alias("n_articles"),
             F.round(F.avg("WikiRank_score"), 2).alias("avg_score"))
        .orderBy(F.desc("n_articles")))
q1.coalesce(1).write.mode("overwrite").parquet(RESULTS + "/q1_stats_by_language")
print(f"P1: {time.time() - t:.1f} s")

q1 = spark.read.parquet(RESULTS + "/q1_stats_by_language")
q1.orderBy(F.desc("n_articles")).show(60)

P1: 2.3 s
+----------+----------+---------+
|  Language|n_articles|avg_score|
+----------+----------+---------+
|        en|   6860430|    24.71|
|        de|   2930971|    18.63|
|        fr|   2626210|    16.12|
|        sv|   2591505|    20.02|
|        nl|   2163715|    12.73|
|        ru|   1991787|    18.18|
|        es|   1905093|    18.77|
|        it|   1875385|    13.63|
|        pl|   1624175|    17.03|
|        zh|   1433026|    20.31|
|        ja|   1424380|    14.57|
|        uk|   1339146|    18.82|
|        vi|   1294722|    14.24|
|        ar|   1237943|    17.11|
|        pt|   1130161|    17.71|
|        fa|   1008969|    11.43|
|        ca|    756348|    21.16|
|        id|    696238|    13.93|
|        sr|    692273|     15.9|
|        ko|    679238|    10.44|
|        no|    633769|    15.53|
|        tr|    617404|    18.14|
|        fi|    577535|    17.64|
|        cs|    550666|    16.44|
|        hu|    544427|    19.04|
|        ro|    464908|    16.61|
|   

The same was implemented using Spark SQL.

In [14]:
articles.createOrReplaceTempView("articles")
spark.sql("""
    SELECT Language, COUNT(*) AS n_articles, ROUND(AVG(WikiRank_score), 2) AS avg_score
    FROM articles
    GROUP BY Language
    ORDER BY n_articles DESC
""").show(5)


+--------+----------+---------+
|Language|n_articles|avg_score|
+--------+----------+---------+
|      en|   6860430|    24.71|
|      de|   2930971|    18.63|
|      fr|   2626210|    16.12|
|      sv|   2591505|    20.02|
|      nl|   2163715|    12.73|
+--------+----------+---------+
only showing top 5 rows


**Pregunta 2:** Top 10 de cada idioma

In [15]:
w = Window.partitionBy("Language").orderBy(F.desc("WikiRank_score"), F.asc("Page_ID"))

t = time.time()
q2 = (articles
        .withColumn("rank", F.row_number().over(w))
        .filter(F.col("rank") <= 10)
        .select("Language", "rank", "Title", "WikiRank_score", "Page_ID"))
q2.coalesce(1).write.mode("overwrite").parquet(RESULTS + "/q2_top10_by_language")
print(f"P2: {time.time() - t:.1f} s")

q2 = spark.read.parquet(RESULTS + "/q2_top10_by_language")
q2.filter("Language = 'es'").orderBy("rank").show(truncate=False)


P2: 42.0 s
+--------+----+---------------------+--------------+-------+
|Language|rank|Title                |WikiRank_score|Page_ID|
+--------+----+---------------------+--------------+-------+
|es      |1   |Apolo                |100.0         |293    |
|es      |2   |Budismo              |100.0         |317    |
|es      |3   |Brasil               |100.0         |410    |
|es      |4   |Camino de Santiago   |100.0         |765    |
|es      |5   |Heráclito            |100.0         |1429   |
|es      |6   |Imperio romano       |100.0         |1509   |
|es      |7   |Japón                |100.0         |1575   |
|es      |8   |Johann Sebastian Bach|100.0         |1591   |
|es      |9   |Orchidaceae          |100.0         |2062   |
|es      |10  |Pablo Picasso        |100.0         |2147   |
+--------+----+---------------------+--------------+-------+



In [16]:
articles.filter("WikiRank_score = 100").groupBy("Language").count().orderBy(F.desc("count")).show(10)

+--------+-----+
|Language|count|
+--------+-----+
|      en| 7804|
|      de| 1930|
|      zh| 1045|
|      es|  749|
|      fr|  630|
|      ru|  604|
|      pt|  580|
|      ms|  508|
|      ca|  426|
|      pl|  409|
+--------+-----+
only showing top 10 rows


**Pregunta 3:** ¿Dependen el tamaño y la calidad de una Wikipedia de las características de su lengua (hablantes, escritura, familia y dificultad FSI)?

In [17]:
q1 = spark.read.parquet(RESULTS + "/q1_stats_by_language")
lang_level = (q1.join(lang_info, "Language")
    .filter(~F.col("Language").isin("simple", "sh", "la", "eo", "vo"))
    .withColumn("articles_per_million", F.round(F.col("n_articles") / F.col("speakers_millions")).cast("long"))
    .withColumn("log_speakers", F.log10("speakers_millions"))
    .withColumn("log_articles", F.log10("n_articles")))

(lang_level.select("Language", "language_name", "speakers_millions", "n_articles",
                   "articles_per_million", "avg_score")
    .coalesce(1).write.mode("overwrite").parquet(RESULTS + "/q3_speakers_by_language"))
spark.read.parquet(RESULTS + "/q3_speakers_by_language") \
     .orderBy(F.desc("articles_per_million")).show(55, truncate=False)

corr_lang = lang_level.agg(
    F.count("*").alias("n_languages"),
    F.round(F.covar_samp("log_speakers", "log_articles"), 3).alias("covar_speakers_articles"),
    F.round(F.corr("log_speakers", "log_articles"), 3).alias("corr_speakers_articles"),
    F.round(F.corr("log_speakers", "avg_score"), 3).alias("corr_speakers_score"))
corr_lang.coalesce(1).write.mode("overwrite").parquet(RESULTS + "/q3_speakers")
corr_lang.show()

+----------+-----------------+-----------------+----------+--------------------+---------+
|Language  |language_name    |speakers_millions|n_articles|articles_per_million|avg_score|
+----------+-----------------+-----------------+----------+--------------------+---------+
|eu        |Basque           |1.034            |430677    |416515              |15.76    |
|et        |Estonian         |0.854            |246512    |288656              |18.07    |
|sv        |Swedish          |13.338           |2591505   |194295              |20.02    |
|nn        |Norwegian Nynorsk|1.377            |170159    |123572              |18.87    |
|no        |Norwegian        |5.51             |633769    |115022              |15.53    |
|fi        |Finnish          |5.789            |577535    |99764               |17.64    |
|sl        |Slovenian        |1.967            |186122    |94622               |17.87    |
|ca        |Catalan          |8.219            |756348    |92024               |21.16    |

In [18]:
arts = articles.join(F.broadcast(lang_info.select("Language", "script", "family", "fsi_category")),
                     "Language", "left")

by_dim = arts.select("Language", "WikiRank_score",
    F.expr("stack(3, 'script', script, 'family', family, "
           "'fsi_category', coalesce(cast(fsi_category as string), 'sin clasificar')) as (dimension, value)"))

t = time.time()
q3_traits = (by_dim.groupBy("dimension", "value")
    .agg(F.countDistinct("Language").alias("n_languages"),
         F.count("*").alias("n_articles"),
         F.round(F.avg("WikiRank_score"), 2).alias("avg_score"),
         F.percentile_approx("WikiRank_score", 0.5, 10000).alias("median_score"),
         F.percentile_approx("WikiRank_score", 0.9, 10000).alias("p90_score"),
         F.round(F.avg((F.col("WikiRank_score") >= 50).cast("int")) * 100, 1).alias("pct_score_50")))
q3_traits.coalesce(1).write.mode("overwrite").parquet(RESULTS + "/q3_by_language_traits")
print(f"P3b: {time.time() - t:.1f} s")

res = spark.read.parquet(RESULTS + "/q3_by_language_traits")
for d in ["script", "family", "fsi_category"]:
    res.filter(F.col("dimension") == d).orderBy(F.desc("n_articles")).show(truncate=False)

P3b: 93.0 s
+---------+-----+-----------+----------+---------+------------+---------+------------+
|dimension|value|n_languages|n_articles|avg_score|median_score|p90_score|pct_score_50|
+---------+-----+-----------+----------+---------+------------+---------+------------+
|script   |Latn |36         |34633194  |18.34    |17.0282     |34.1325  |3.0         |
|script   |Cyrl |6          |4807894   |17.48    |17.134      |30.8061  |1.4         |
|script   |Arab |3          |2458050   |14.51    |13.7343     |25.4356  |0.6         |
|script   |Hans |1          |1433026   |20.31    |17.8323     |40.6602  |5.3         |
|script   |Jpan |1          |1424380   |14.57    |11.8164     |29.5567  |1.8         |
|script   |Kore |1          |679238    |10.44    |6.5515      |25.0323  |0.9         |
|script   |Hebr |1          |359029    |21.8     |19.8936     |41.0657  |4.9         |
|script   |Armn |1          |312200    |18.05    |19.391      |29.8919  |1.2         |
|script   |Grek |1          |23

**Machine Learning:** regresión de WikiRank_score (MLlib)

ML-1: tabla auxiliar de palabras

In [19]:
TITLE_WORDS = HDFS + cfg["paths"]["title_words"]
min_count = cfg["title_words"]["min_count"]
tokens = F.explode(F.split(F.lower("Title"), r"[^\p{L}\p{N}]+")).alias("word")

t = time.time()
title_words = (articles.select("Language", "Page_ID", tokens)
    .filter(F.length("word") > 0)
    .groupBy("Language", "word")
    .agg(F.countDistinct("Page_ID").alias("n_titles"))
    .filter(F.col("n_titles") >= min_count))
(title_words.repartition(part)
    .write.mode("overwrite").partitionBy(part).parquet(TITLE_WORDS))
title_words = spark.read.parquet(TITLE_WORDS)
print(f"Tabla de palabras: {time.time() - t:.0f} s | filas: {title_words.count()}")
title_words.filter("Language = 'es'").orderBy(F.desc("n_titles")).show(10)

Tabla de palabras: 237 s | filas: 2219634
+---------+--------+--------+
|     word|n_titles|Language|
+---------+--------+--------+
|       de|  378170|      es|
|       la|   64246|      es|
|      del|   46808|      es|
|       en|   32707|      es|
|      los|   26402|      es|
|       el|   25176|      es|
|        y|   24601|      es|
|municipio|   23645|      es|
|      the|   20794|      es|
|      san|   18660|      es|
+---------+--------+--------+
only showing top 10 rows


ML-2: datos del modelo

In [20]:
m = cfg["ml"]
sample = articles.sample(m["sample_fraction"], seed=m["seed"])
n_lang = spark.read.parquet(RESULTS + "/q1_stats_by_language").select("Language", "n_articles")

word_feats = (sample.select("Language", "Page_ID", tokens)
    .filter(F.length("word") > 0)
    .join(title_words, ["Language", "word"], "left")
    .fillna({"n_titles": 1})
    .join(F.broadcast(n_lang), "Language")
    .withColumn("word_log_freq", F.log10(F.col("n_titles") / F.col("n_articles")))
    .groupBy("Language", "Page_ID")
    .agg(F.avg("word_log_freq").alias("words_avg_freq")))

w_lang = Window.partitionBy("Language").orderBy("Page_ID")
data = (sample
    .join(word_feats, ["Language", "Page_ID"], "left")
    .join(F.broadcast(n_lang), "Language")
    .withColumn("words_avg_freq", F.coalesce("words_avg_freq", F.log10(1 / F.col("n_articles"))))    
    .withColumn("page_id_pct", F.percent_rank().over(w_lang))
    .withColumn("title_length", F.length("Title").cast("double"))
    .withColumn("title_has_year", F.col("Title").rlike(r"\b(1[0-9]{3}|20[0-2][0-9])\b").cast("double"))
    .withColumn("title_has_paren", F.col("Title").contains("(").cast("double"))
    .withColumn("title_has_comma", F.col("Title").contains(",").cast("double"))
    .select("Language", "WikiRank_score", "page_id_pct", "title_length", "title_has_year",
            "title_has_paren", "title_has_comma", "words_avg_freq"))
data.cache()
print("Filas para el modelo:", data.count())

Filas para el modelo: 2352462


ML-3: comprobación de las variables

In [21]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd
pd.set_option("display.width", 200)
numeric = ["page_id_pct", "title_length", "title_has_year", "title_has_paren", "title_has_comma", "words_avg_freq"]

vec = VectorAssembler(inputCols=numeric, outputCol="v").transform(data).select("v")
corr = Correlation.corr(vec, "v").head()[0].toArray()
print(pd.DataFrame(corr, index=numeric, columns=numeric).round(2))

w = Window.partitionBy("Language")
check = data.select("Language", *[F.avg(c).over(w).alias(c + "_m") for c in numeric], *numeric)
row = check.agg(*[F.var_pop(c).alias(c) for c in numeric],
                *[F.var_pop(c + "_m").alias(c + "_m") for c in numeric]).first()
for c in numeric:
    print(f"{c:16s} variación explicada por la lengua: {100 * row[c + '_m'] / row[c]:.1f} %")

cand = sample.select(F.length("Title").alias("len"),
                     F.size(F.split(F.trim("Title"), r"\s+")).alias("words"))
print("corr(longitud, nº de palabras):", round(cand.stat.corr("len", "words"), 3))

                 page_id_pct  title_length  title_has_year  title_has_paren  title_has_comma  words_avg_freq
page_id_pct             1.00          0.11            0.07             0.04             0.01            0.04
title_length            0.11          1.00            0.32             0.25             0.18            0.53
title_has_year          0.07          0.32            1.00             0.07             0.01            0.24
title_has_paren         0.04          0.25            0.07             1.00             0.02            0.24
title_has_comma         0.01          0.18            0.01             0.02             1.00            0.12
words_avg_freq          0.04          0.53            0.24             0.24             0.12            1.00
page_id_pct      variación explicada por la lengua: 0.0 %
title_length     variación explicada por la lengua: 9.7 %
title_has_year   variación explicada por la lengua: 1.1 %
title_has_paren  variación explicada por la lengua: 4.0 %
title

ML-4: entrenamiento

In [22]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression

idx_lang = StringIndexer(inputCol="Language", outputCol="lang_idx", stringOrderType="frequencyAsc")
encoder = OneHotEncoder(inputCols=["lang_idx"], outputCols=["lang_vec"])
assembler = VectorAssembler(inputCols=numeric + ["lang_vec"], outputCol="features")
lr = LinearRegression(featuresCol="features", labelCol="WikiRank_score")

train, test = data.randomSplit([m["train_ratio"], 1 - m["train_ratio"]], seed=m["seed"])
t = time.time()
model = Pipeline(stages=[idx_lang, encoder, assembler, lr]).fit(train)
print(f"Entrenado en {time.time() - t:.0f} s")

Entrenado en 14 s


ML-5: evaluación frente al baseline

In [23]:
from pyspark.ml.evaluation import RegressionEvaluator
pred = model.transform(test)
ev = RegressionEvaluator(labelCol="WikiRank_score", predictionCol="prediction")
rmse = ev.evaluate(pred, {ev.metricName: "rmse"})
mae = ev.evaluate(pred, {ev.metricName: "mae"})
r2 = ev.evaluate(pred, {ev.metricName: "r2"})

mean_train = train.agg(F.avg("WikiRank_score")).first()[0]
rmse_base = test.select(F.sqrt(F.avg((F.col("WikiRank_score") - mean_train) ** 2))).first()[0]

metrics = spark.createDataFrame([(rmse, mae, r2, rmse_base)], ["rmse", "mae", "r2", "rmse_baseline_media"])
metrics.coalesce(1).write.mode("overwrite").json(RESULTS + "/ml_metrics")
metrics.show()

coefs = model.stages[-1].coefficients.toArray()[:len(numeric)]
print(pd.DataFrame({"variable": numeric, "coeficiente": coefs.round(3)}))
(spark.createDataFrame(list(zip(numeric, coefs.tolist())), ["variable", "coeficiente"])
     .coalesce(1).write.mode("overwrite").json(RESULTS + "/ml_coefficients"))

no_words = [c for c in numeric if c != "words_avg_freq"]
assembler_nw = VectorAssembler(inputCols=no_words + ["lang_vec"], outputCol="features")
model_nw = Pipeline(stages=[idx_lang, encoder, assembler_nw, lr]).fit(train)
rmse_nw = ev.evaluate(model_nw.transform(test), {ev.metricName: "rmse"})
print(f"RMSE con words_avg_freq: {rmse:.3f} | sin: {rmse_nw:.3f} | "
      f"mejora: {100 * (rmse_nw - rmse) / rmse_nw:.1f} %")

_ = data.unpersist()

+------------------+-----------------+-------------------+-------------------+
|              rmse|              mae|                 r2|rmse_baseline_media|
+------------------+-----------------+-------------------+-------------------+
|13.188418237050794|9.881907611861987|0.08144684247447465| 13.760701416068573|
+------------------+-----------------+-------------------+-------------------+

          variable  coeficiente
0      page_id_pct        1.750
1     title_length        0.058
2   title_has_year       -1.141
3  title_has_paren       -1.150
4  title_has_comma       -1.055
5   words_avg_freq        0.406
RMSE con words_avg_freq: 13.188 | sin: 13.193 | mejora: 0.0 %


In [24]:
spark.stop()